In [1]:
import os
import numpy as np
import glob
import matplotlib.pyplot as plt
from ase.io import read, write
import pandas as pd
from ase import Atoms
from ase.neighborlist import NeighborList, natural_cutoffs
from itertools import combinations
import networkx as nx
import math
from ase.data import covalent_radii

## Reading OCV and Capacity

In [2]:
data_1 = pd.read_excel('./dice_wOCV_wcap.xlsx')
data_2 = pd.read_excel('./dice_wOCV_wcap.xlsx',sheet_name = 'Final Stage')

In [3]:
data_ccof = pd.read_excel('./CORE-COF-2DCOF-1099.xlsx')
ccof_tags = np.array(data_ccof.select_dtypes(int))
ccof_feats = data_ccof.select_dtypes(float)

In [4]:
ccof_tags.flatten()

array([   1,    2,    3, ..., 1240, 1241, 1242])

In [5]:
ccof_names = data_ccof["Number"]

## OCV and capacity data preparation

In [6]:
cofid_fp = data_1["COF_numbers"]
nli_fp = data_1["No. of Li"]
vol_fp = data_1["Voltage"]
cap_fp = data_1["Capacity"]
mw_fp = data_1["molecular weight"]

In [7]:
cofid_lp = data_2["COF_numbers"]
nli_lp = data_2["No. of Li"]
vol_lp = data_2["Voltage"]
cap_lp = data_2["Capacity"]
mw_lp = data_2["molecular weight"]

In [8]:
vol_fp.max(),vol_fp.min(),vol_fp.mean(), cofid_fp[np.argmax(vol_fp)], cofid_fp[np.argmin(vol_fp)]

(2.71501359000001, -0.269747046666667, 1.023353147592761, 254, 182)

In [9]:
cap_lp.max(),cap_lp.min(),cap_lp.mean(), cofid_lp[np.argmax(cap_lp)], cofid_lp[np.argmin(cap_lp)]

(700.013059945148, 34.7260577275359, 292.71049505886145, 183, 1113)

In [10]:
# overlap between the DICE-C and DICE-V sets
common_count = len(set(cofid_fp) & set(cofid_lp))
print(common_count)
diff = set(cofid_fp) ^ set(cofid_lp)
print(diff)
only_in_list2 = set(cofid_lp) - set(cofid_fp)
print("For DICE-C",only_in_list2)

134
{1096, 621, 178, 182, 235, 955}
For DICE-C {1096, 178, 235}


## Duplicates check

In [12]:
def list_duplicates(seq):
  seen = set()
  seen_add = seen.add
  # adds all elements it doesn't know yet to seen and all other to seen_twice
  seen_twice = set( x for x in seq if x in seen or seen_add(x) )
  # turn the set into a list (as requested)
  return list( seen_twice )

In [13]:
list_duplicates(cofid_lp)

[]

In [14]:
len(cofid_fp), len(cofid_lp)

(137, 137)

## FEATURES

In [15]:
from utils import *

In [16]:
def global_chem_feats(atoms):
    symbols = atoms.get_chemical_symbols()
    natoms = len(symbols)
    o_indices = []
    n_indices = []
    c_indices = []
    h_indices = []
    f_indices = []
    cl_indices = []
    br_indices = []
    for idx, elements in enumerate(symbols):
        if elements == "O":
            o_indices.append(idx)
        if elements == "N":
            n_indices.append(idx)
        if elements == "C":
            c_indices.append(idx)
        if elements == "H":
            h_indices.append(idx)
        if elements == "F":
            f_indices.append(idx)
        if elements == "Cl":
            cl_indices.append(idx)
        if elements == "Br":
            br_indices.append(idx)
    n_c = len(c_indices)
    n_h = len(h_indices)
    n_n = len(n_indices)
    n_o = len(o_indices)
    n_f = len(f_indices)
    n_cl = len(cl_indices)
    n_br = len(br_indices)
    frac_c = n_c/natoms
    frac_n = n_n/natoms
    frac_o = n_o/natoms
    frac_no = (n_o+n_n)/natoms
    frac_h = n_h/natoms
    frac_hal = (n_cl + n_f + n_br)/natoms
    tdu = ((2 *(n_c + 0.5 * n_n) + 2 - (n_h + n_f + n_cl + n_br)))/2
    te = (n_o + n_n + n_f + n_cl + n_br)/natoms
    return frac_c, frac_h, frac_n, frac_o, frac_no, frac_hal, tdu, te


## Some functions to locate the rank-ordered RAFs

In [17]:
#carboxylates COOH
def identify_carboxylate_groups(atoms):
    """
    identify carboxylate groups in an ASE Atoms object,
    using presence/absence of hydrogen bonded to oxygen to determine the carbonyl oxygen.

    returns:
        count: number of COO groups
        carboxy_carbons: list of carbon atom indices
        carbonyl_oxygens: list of oxygen indices considered as carbonyl
    """
    cutoffs = natural_cutoffs(atoms)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(atoms)

    carboxy_carbons = []
    carbonyl_oxygens = []

    for c_index, atom in enumerate(atoms):
        if atom.symbol != 'C':
            continue

        neighbors = nl.get_neighbors(c_index)[0]
        o_neighbors = [i for i in neighbors if atoms[i].symbol == 'O']

        if len(o_neighbors) == 2:
            # Determine which O is not bonded to H (likely the carbonyl)
            o_no_H = []
            for o_index in o_neighbors:
                o_nbrs = nl.get_neighbors(o_index)[0]
                bonded_H = any(atoms[n].symbol == 'H' for n in o_nbrs)
                if not bonded_H:
                    o_no_H.append(o_index)

            # Only accept if exactly one oxygen has no H bonded (i.e., is carbonyl)
            if len(o_no_H) == 1:
                carboxy_carbons.append(c_index)
                carbonyl_oxygens.append(o_no_H[0])

    count = len(carboxy_carbons)
    return count, carboxy_carbons, carbonyl_oxygens

In [18]:
#bicarboxylicimides NCOCO
def identify_bicarboxylic_imides_v2(atoms):
    """
    identify bicarboxylic imide groups:
    - N bonded to at least 2 carbon atoms,
    - Each of 2 carbon neighbors is bonded to a carbonyl O:
        - O has coordination number 1,
        - O bonded only to C,
        - O not bonded to H.
    
    returns:
        count: number of imide groups
        imide_nitrogens: list of N atom indices
        carbonyl_oxygens: list of tuples (O1_idx, O2_idx) per group
    """
    cutoffs = natural_cutoffs(atoms)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(atoms)

    imide_nitrogens = []
    carbonyl_oxygens = []

    for n_idx, atom in enumerate(atoms):
        if atom.symbol != 'N':
            continue

        n_neighbors = nl.get_neighbors(n_idx)[0]
        c_neighbors = [i for i in n_neighbors if atoms[i].symbol == 'C']
        if len(c_neighbors) < 2:
            continue

        # Try all C–C combinations
        for c1_idx, c2_idx in combinations(c_neighbors, 2):
            o_indices = []

            for c_idx in [c1_idx, c2_idx]:
                c_neighbors = nl.get_neighbors(c_idx)[0]
                carbonyl_o = None

                for o_idx in c_neighbors:
                    if atoms[o_idx].symbol != 'O':
                        continue
                    o_nbrs = nl.get_neighbors(o_idx)[0]

                    # Carbonyl oxygen must:
                    if (len(o_nbrs) == 1 and atoms[o_nbrs[0]].symbol == 'C'):
                        carbonyl_o = o_idx
                        break

                if carbonyl_o is None:
                    break  # This carbon didn't have a valid carbonyl O
                o_indices.append(carbonyl_o)

            if len(o_indices) == 2:
                imide_nitrogens.append(n_idx)
                carbonyl_oxygens.append(o_indices)
                break  # Only report first match per nitrogen
    motifs = [[n] + o for n, o in zip(imide_nitrogens, carbonyl_oxygens)]
    o_ncoco = [item for sublist in carbonyl_oxygens for item in sublist]

    return len(imide_nitrogens), motifs, o_ncoco

In [19]:
## identify porphyrins
def identify_porphyrin_units_by_proximity(atoms, max_n_distance=4.5):
    """
    identify porphyrin-like cores based only on proximity of 4 nitrogen atoms.

    args:
        atoms: ASE Atoms object
        max_n_distance: Maximum allowed N–N distance (in Å)

    returns:
        count: number of porphyrins
        porphyrin_nitrogens: list of tuples (4 N atom indices)
    """
    # Get all nitrogen atom indices
    n_indices = [i for i, atom in enumerate(atoms) if atom.symbol == 'N']
    positions = atoms.positions[n_indices]

    porphyrin_units = []

    for combo in combinations(range(len(n_indices)), 4):
        idxs = [n_indices[i] for i in combo]
        coords = atoms.positions[idxs]
        
        # Compute all pairwise distances
        dists = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)
        #print(dists)
        max_dist = np.max(dists)

        if max_dist <= max_n_distance:
            porphyrin_units.append(idxs)

    return len(porphyrin_units), porphyrin_units

In [20]:
##-C(=O)NN

def identify_conn_motifs(atoms):
    """
    identify -C(=O)N-N motifs in ASE Atoms

    returns:
        count: number of motifs found
        motifs: list of tuples (C, O, N1, N2) atom indices
    """
    cutoffs = natural_cutoffs(atoms)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(atoms)

    motifs = []

    for c_idx, atom in enumerate(atoms):
        if atom.symbol != 'C':
            continue

        neighbors = nl.get_neighbors(c_idx)[0]
        o_neighbors = [i for i in neighbors if atoms[i].symbol == 'O']
        n_neighbors = [i for i in neighbors if atoms[i].symbol == 'N']

        # Look for exactly one oxygen and one nitrogen neighbor
        if len(o_neighbors) < 1 or len(n_neighbors) < 1:
            continue

        # Pick first O not bonded to H (likely carbonyl)
        carbonyl_O = None
        for o_idx in o_neighbors:
            o_nbrs = nl.get_neighbors(o_idx)[0]
            if not any(atoms[j].symbol == 'H' for j in o_nbrs):
                carbonyl_O = o_idx
                break

        if carbonyl_O is None:
            continue

        for n1_idx in n_neighbors:
            # Now check if this N is bonded to another N
            n1_neighbors = nl.get_neighbors(n1_idx)[0]
            n2_candidates = [i for i in n1_neighbors if atoms[i].symbol == 'N' and i != c_idx]

            for n2_idx in n2_candidates:
                motifs.append([c_idx, carbonyl_O, n1_idx, n2_idx])

    o_conn = [sublist[1] for sublist in motifs]

    return len(motifs), motifs, o_conn

In [21]:
# nitro NO2

def identify_nitro_groups(atoms, max_oh_bond_distance=1.2):
    """
    identify -NO2 (nitro) groups in an ASE Atoms object.

    returns:
        count: number of nitro groups
        nitro_nitrogens: list of N atom indices
        nitro_oxygens: list of tuples (O1_idx, O2_idx) per group
    """
    cutoffs = natural_cutoffs(atoms)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(atoms)

    nitro_nitrogens = []
    nitro_oxygens = []

    for n_idx, atom in enumerate(atoms):
        if atom.symbol != 'N':
            continue

        neighbors = nl.get_neighbors(n_idx)[0]
        o_neighbors = [i for i in neighbors if atoms[i].symbol == 'O']

        if len(o_neighbors) != 2:
            continue  # Not a nitro group

        valid_o = []
        for o_idx in o_neighbors:
            # Check for H atoms within O–H bond distance
            is_bonded_to_H = False
            for j, other_atom in enumerate(atoms):
                if other_atom.symbol == 'H':
                    dist = atoms.get_distance(o_idx, j, mic=True)
                    if dist <= max_oh_bond_distance:
                        is_bonded_to_H = True
                        break
            if not is_bonded_to_H:
                valid_o.append(o_idx)

        if len(valid_o) == 2:
            nitro_nitrogens.append(n_idx)
            nitro_oxygens.append(valid_o)
    motifs = [[n] + o for n, o in zip(nitro_nitrogens, nitro_oxygens)]
    o_ncoco = [item for sublist in nitro_oxygens for item in sublist]

    return len(nitro_nitrogens), motifs, nitro_oxygens

In [22]:
### for sulphonyl groups

def identify_sulfonic_acid_groups(atoms, max_oh_distance=1.2):
    """
    identify -SO3H (sulfonic acid) groups in an ASE Atoms object.

    Returns:
        count: number of -SO3H groups
        sulfur_indices: list of S atom indices
        group_oxygens: list of tuples (OH_O_idx, O1_idx, O2_idx)
    """
    cutoffs = natural_cutoffs(atoms)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(atoms)

    sulfonic_s = []
    group_oxygens = []

    for s_idx, atom in enumerate(atoms):
        if atom.symbol != 'S':
            continue

        s_neighbors = nl.get_neighbors(s_idx)[0]
        o_neighbors = [i for i in s_neighbors if atoms[i].symbol == 'O']

        if len(o_neighbors) != 3:
            continue  # Must be bonded to exactly 3 O atoms

        oh_o = None
        non_oh_os = []

        for o_idx in o_neighbors:
            # Check if O is bonded to a hydrogen (within OH bond distance)
            is_oh = False
            for j, other_atom in enumerate(atoms):
                if other_atom.symbol == 'H':
                    dist = atoms.get_distance(o_idx, j, mic=True)
                    if dist <= max_oh_distance:
                        is_oh = True
                        break

            if is_oh and oh_o is None:
                oh_o = o_idx
            elif not is_oh:
                non_oh_os.append(o_idx)

        if oh_o is not None and len(non_oh_os) == 2:
            sulfonic_s.append(s_idx)
            group_oxygens.append([oh_o, non_oh_os[0], non_oh_os[1]])

    return len(sulfonic_s), np.array(group_oxygens).flatten()


## loading pacmof for coordination shell features

In [23]:
from pacmof import get_features_from_cif_serial

## EN weighted RDFs

In [24]:
from elementdata import ElementData
ed = ElementData()
ve = ed.valenceelectrons
rad = ed.CovalentRadius2
en_pau = ed.ElectroNegativityPauling

In [25]:
def determin_prox_sites_aprdf(struc,G,sp2_o,c_r,B):
    #if any(struc.get_pbc()):
    #    struc.set_pbc(False)
    symbols = struc.get_chemical_symbols()
    numbers = np.array(struc.get_atomic_numbers())
    #c_r = 2.5
    BOB = []
    for idx, elements in enumerate(symbols):
        #if elements == "O" or elements == "N" or elements == "S":
        #    ons.append(idx)
    ## new neighborlist
        if idx in sp2_o:
            BOB.append(c_r)
        else:
            BOB.append(covalent_radii[numbers[idx]])
    #print(BOB)
    nl = NeighborList(cutoffs=BOB, bothways=True, self_interaction=False)
    nl.update(struc)
    #print(ons)
    elems = ['N', 'O', 'S']
    fp_feats_o = []
    m = 0
    #B=100
    rdf=np.zeros([10])
    R=np.linspace(2,6,10)
    if len(sp2_o) != 0:
        for idx in sp2_o:
            pr2o = []
            pr2o_en = []
            pr2o_d = []
            nei, _ = nl.get_neighbors(idx)
            print("The neighborlist of", idx, "is", nei)
            for i in nei:
                for l in range(R.size):
                    rdf[l]+= en_pau[symbols[idx]]* en_pau[symbols[i]]*math.exp(-B*(struc.get_distance(idx,i)-R[l])**2)
                if symbols[i] in elems:
                    try:
                        path = len(nx.shortest_path(G, source=idx, target = i))
                        if path >=4:
                            print("proximal atom found between", idx, "and", i)
                            pr2o.append(i)
                            pr2o_en.append(en_pau[symbols[i]])
                            pr2o_d.append(struc.get_distance(idx,i))
                    except:
                        print("proximal atom found between", idx, "and", i)
                        pr2o.append(i)
                        pr2o_en.append(en_pau[symbols[i]])
                        pr2o_d.append(struc.get_distance(idx,i))
                else:
                    print("not proximal")
            n_pr2o = len(pr2o)
            if n_pr2o != 0:
                mean_pr2o_en = np.mean(pr2o_en)
                max_pr2o_en = np.max(pr2o_en)
                mean_pr2o_d = np.mean(pr2o_d)
                min_pr2o_d = np.min(pr2o_d)
            else:
                mean_pr2o_en = 0.0
                max_pr2o_en = 0.0
                mean_pr2o_d = 10.0
                min_pr2o_d = 10.0
            feats = [n_pr2o, mean_pr2o_en, max_pr2o_en, mean_pr2o_d, min_pr2o_d]
            fp_feats_o.append(feats)
            print(fp_feats_o)
    mean_fp_feats_o = calculate_means(fp_feats_o)
    rdf = [np.round(i, decimals=8) for i in rdf]
    print(rdf)
    return mean_fp_feats_o, rdf

## some useful functions

In [26]:
def calculate_means(arrays):
    # Transpose the list of arrays to separate first, second, and third elements
    transposed = list(zip(*arrays))
    
    # Calculate the bmean for each group of elements
    means = [sum(elements) / len(elements) for elements in transposed]
    
    return means

In [3]:
def csvDf(dat,columns,**kwargs): 
  data = array(dat)
  if data is None or len(data)==0 or len(data[0])==0:
    return None
  else:
    #return pd.DataFrame(data[1:,1:],index=data[1:,0],columns=data[0,1:],**kwargs)
    return pd.DataFrame(data[0:,0:],index=None,columns=columns,**kwargs)

In [28]:
def Agg_mean_feats(idxs,df):
    feats = []
    for idx in idxs:
        feat = [df.info['features'][idx][0],
                df.info['features'][idx][1],
                df.info['features'][idx][2],
                df.info['features'][idx][3],
                df.info['features'][idx][4],
                df.info['features'][idx][5],
                df.info['features'][idx][6]]
    feats.append(feat)
    mean_feats = calculate_means(feats)
    return mean_feats
def lcf_maker(idxs,mean_feats):
    lcf = np.zeros(8)
    lcf[0] = len(idxs)
    for j in range(1,8):
        lcf[j] = mean_feats[j-1]
    return lcf
def lcf_maker_motif(n,motifs,df):
    feats = []
    for motif in motifs:
        feat = Agg_mean_feats(motif,df)
        feats.append(feat)
    mean_feats = calculate_means(feats)
    lcf = np.zeros(8)
    lcf[0] = n
    for j in range(1,8):
        lcf[j] = mean_feats[j-1]
    return lcf

## Featurizing DICE-V dataset

In [54]:
## dice-v features
gl_chem_feats = []
l_chem_feats = []
l_fp_feats = []
l_rdf_feats = []
count_o = 0
count_n = 0
count_r = 0
cofs_ncoco = []
cofs_conn = []
cofs_cooh = []
cofs_so3h = []
cofs_por=[]
cofs_nitro=[]
c_r = 2.0
B = 100
for i in cofid_fp:
    struc = read(f'./v-predict-1p/cifs/{i}.cif')
    gl_chem_feats.append(global_chem_feats(struc))
    #l_chem_feats.append(local_chem_feats(struc))
    nl, cm = compute_ase_neighbour(struc)
    graph = matrix2dict(cm)
    cutoffs = natural_cutoffs(struc, mult=1.0)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(struc)
    G = nx.from_numpy_array(cm)
    #struc analysis
    sp2_o = find_NO(struc)[0]
    sp2_n = find_NO(struc)[1]
    six_rings = find_rings(G, 6)
    five_rings = find_rings(G, 5)
    tot_rings = six_rings + five_rings
    ### special groups
    ##### carboxylate
    n_cooh, c_cooh, o_cooh = identify_carboxylate_groups(struc)
    sp2_o_wo_cooh = [x for x in sp2_o if x not in o_cooh] ## remove carboxyl oxygens
    if n_cooh != 0:
        cofs_cooh.append(i)
    ### sulphonyl
    n_so3h, o_so3h = identify_sulfonic_acid_groups(struc)
    only_sp2_o = [x for x in sp2_o_wo_cooh if x not in o_so3h]
    if n_so3h != 0:
        cofs_so3h.append(i)
    ###bisimcooh
    n_ncoco, motif_ncoco, o_ncoco = identify_bicarboxylic_imides_v2(struc)
    if n_ncoco != 0:
        cofs_ncoco.append(i)

    ### conn
    n_conn, motif_conn, o_conn = identify_conn_motifs(struc)
    if n_conn != 0:
        cofs_conn.append(i)

    ### no2 groups

    n_nitro, motif_nitro, o_nitro = identify_nitro_groups(struc)
    if n_nitro != 0:
        cofs_nitro.append(i)
    ### porphyrin
    n_por, motif_por = identify_porphyrin_units_by_proximity(struc)
    if n_por != 0:
        cofs_por.append(i)
    # PACMOF featurizer
    df = get_features_from_cif_serial(f'./v-predict-1p/cifs/{i}.cif')
    lcf_o = np.zeros(8)
    lcf_n = np.zeros(8)
    lcf_rings = np.zeros(8)
    w = 0
    x = 0
    if len(only_sp2_o) !=0:
        if n_ncoco == 0 and n_conn == 0:
            mean_feats_o = Agg_mean_feats(only_sp2_o,df)
            lcf_o = lcf_maker(only_sp2_o,mean_feats_o)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,only_sp2_o,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        elif n_ncoco == 0 and n_conn != 0:
            lcf_o = lcf_maker_motif(n_conn,motif_conn,df)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,o_conn,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        elif n_ncoco != 0 and n_conn == 0:
            lcf_o = lcf_maker_motif(n_ncoco,motif_ncoco,df)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,o_ncoco,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        elif n_ncoco != 0 and n_conn != 0: # prioritize NCOCO
            lcf_o = lcf_maker_motif(n_ncoco,motif_ncoco,df)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,o_ncoco,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        else:
            w = w +1
        print('local feats', l_chem_feats)
    elif len(sp2_n)!=0:
        if n_por == 0:
            mean_feats_n = Agg_mean_feats(sp2_n,df)
            lcf_n = lcf_maker(sp2_n,mean_feats_n)
            l_chem_feats.append(lcf_n)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,sp2_n,2.25,10)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_n = count_n + 1
        elif n_por != 0:
            lcf_n = lcf_maker_motif(2*n_por,motif_por,df)
            l_chem_feats.append(lcf_n)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,np.array(motif_por).flatten(),c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_n = count_n + 1
        else:
            x = x + 1
    else:
        lcf_rings[0] = len(tot_rings)
        lcf_rings[1] = 2.275 ## avg of electronegetivity of C and H
        lcf_rings[2] = 12.429 ## avg of 1st IP of C and H
        lcf_rings[3] = 6 ## numer of atoms in 1st coord shell
        lcf_rings[4] = 2.55 ## numer of atoms in 1st coord shell
        lcf_rings[5] = 2.0 ## avg distance from atoms in 1st coord shell
        lcf_rings[6] = 11.260 ## avg 1st IP in 1st coord shell
        lcf_rings[7] = 2.20 ## avg EN  in 2nd coord shell
        #lcf = np.concatenate((lcf_o,lcf_n,lcf_rings), axis=None)
        l_chem_feats.append(lcf_rings)
        #l_chem_feats.append(lcf)
        l_fp_feats.append([0.0, 0.0, 0.0, 10.0, 10.0])
        l_rdf_feats.append(np.zeros([10]))
        count_r = count_r + 1
    print("finished computing features")

Reading  CIF file ./v-predict-1p/cifs/1049.cif...
Computing features for ./v-predict-1p/cifs/1049.cif...


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 120/120 [00:00<00:00, 658.58it/s]
/home/sdas/miniconda3/envs/oms/lib/python3.12/site-packages/ase/io/cif.py:401: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!
  warnings.warn(


The neighborlist of 4 is [ 34  40 104  16  17   4   1   2   3   4]
not proximal
proximal atom found between 4 and 40
not proximal
not proximal
not proximal
not proximal
not proximal
proximal atom found between 4 and 3
[[2, 3.24, 3.44, 2.3165661206533255, 2.316566120653325]]
The neighborlist of 20 is [33 96 32 20  2  8 17 18 19 20]
not proximal
not proximal
not proximal
not proximal
proximal atom found between 20 and 8
not proximal
not proximal
proximal atom found between 20 and 19
[[2, 3.24, 3.44, 2.3165661206533255, 2.316566120653325], [2, 3.24, 3.44, 2.3165661206533255, 2.3165661206533197]]
The neighborlist of 36 is [100  36   0   1  18  24  33  34  35  36]
not proximal
not proximal
not proximal
not proximal
proximal atom found between 36 and 24
not proximal
not proximal
proximal atom found between 36 and 35
[[2, 3.24, 3.44, 2.3165661206533255, 2.316566120653325], [2, 3.24, 3.44, 2.3165661206533255, 2.3165661206533197], [2, 3.24, 3.44, 2.3165661206533272, 2.3165661206533255]]
The nei

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 108/108 [00:00<00:00, 1341.60it/s]
/home/sdas/miniconda3/envs/oms/lib/python3.12/site-packages/ase/io/cif.py:401: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!
  warnings.warn(


The neighborlist of 26 is [27 31 29 28 30 26 18 21 22 24 26]
not proximal
proximal atom found between 26 and 29
proximal atom found between 26 and 28
not proximal
not proximal
not proximal
not proximal
not proximal
[[2, 3.24, 3.44, 3.677507461714696, 2.840070235290142]]
The neighborlist of 28 is [38 32 29 28 20 23 25 26 27 28]
not proximal
not proximal
proximal atom found between 28 and 29
not proximal
not proximal
not proximal
proximal atom found between 28 and 26
[[2, 3.24, 3.44, 3.677507461714696, 2.840070235290142], [2, 3.24, 3.44, 3.5602285633868043, 2.605512438634358]]
The neighborlist of 62 is [64 63 65 66 67 62 54 57 58 60 62]
proximal atom found between 62 and 64
proximal atom found between 62 and 65
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
[[2, 3.24, 3.44, 3.677507461714696, 2.840070235290142], [2, 3.24, 3.44, 3.5602285633868043, 2.605512438634358], [2, 3.24, 3.44, 3.6775074617146997, 2.8400702352901477]]
The neighborlist of 64 is [68 65 7

...

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 180/180 [00:00<00:00, 1225.28it/s]


The neighborlist of 3 is [ 31 173  14 159  15  22   4   5  16   3  22   4   5  16   0   0   1   2
   2   3]
not proximal
not proximal
not proximal
proximal atom found between 3 and 159
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
[[1, 3.44, 3.44, 2.6444827950175966, 2.6444827950175966]]
The neighborlist of 33 is [ 61  45  46 113  44  34  35  52  99  33  46  34  35  52  30  30  31  32
  32  33]
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
proximal atom found between 33 and 99
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
[[1, 3.44, 3.44, 2.6444827950175966, 2.6444827950175966], [1, 3.44, 3.44, 2.6444827950176073, 2.6444827950176073]]
The neighborlist of 63 is [ 74  64 129 143  65  82  76  75  64  63  65  82  76   1  60  60  61  62
  62  63]
not proximal
proximal atom found be

100%|████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [00:00<00:00, 1385.40it/s]

The neighborlist of 54 is [56 25  3 48 54 19 25  3 48  0 18 18 19 23 24 54]
proximal atom found between 54 and 56
not proximal
not proximal
proximal atom found between 54 and 48
not proximal
not proximal
not proximal
proximal atom found between 54 and 48
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
[[3, 3.1733333333333333, 3.44, 7.677686760021429, 3.236751901763232]]
The neighborlist of 55 is [55  2 17 17 21 22 23 24 26 53 55]
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
proximal atom found between 55 and 53
[[3, 3.1733333333333333, 3.44, 7.677686760021429, 3.236751901763232], [1, 3.04, 3.04, 17.65315970313245, 17.65315970313245]]
The neighborlist of 56 is [25  1 56 20  4  4 19 20 21 26 49 54 56]
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
proximal atom found between 56 and 49
proximal atom found between 56 and 54
[[3, 3.17

KeyboardInterrupt: 

In [ ]:
## Assembing the features
data1 = csvDf(gl_chem_feats, ['C','H', 'N', 'O', 'NO', 'halo', 'tdu', 'te'])
data2 = csvDf(l_chem_feats, ['a','b','c','d','e','f','g','h'])
data3 = csvDf(l_fp_feats, ['mean_num','mean_en', 'max_en', 'mean_d', 'max_d'])
data4 = csvDf(l_rdf_feats, ['aa','bb','cc','dd','ee','ff','gg','hh','ii', 'jj'])
labels = []
for i in vol_fp:
    if  i >= 1.6:
        labels.append(1)
    else:
        labels.append(0)
data_t = pd.DataFrame(labels, columns = ['class'])
data = pd.concat([data1,
                  data2,
                  data3,
                 data4,
                  data_t
                 ], 
                 axis=1)
np.save("voltage_data.npy",data.to_numpy())

## Featurizing full Core COF database for prediction

In [31]:
ccofs = glob.glob("./CORE-COF-2DCOF-997/*cif")
ccofs = [i[22:-4] for i in ccofs]

In [32]:
## core cof features
gl_chem_feats = []
l_chem_feats = []
l_fp_feats = []
l_rdf_feats = []
misc_feats = []
lattice_params = []
mw_ccof = []
count_o = 0
count_n = 0
count_r = 0
cofs_ncoco = []
cofs_conn = []
cofs_cooh = []
cofs_so3h = []
cofs_por=[]
cofs_nitro=[]
c_r = 2.0
B = 100
for i in ccofs:
    struc = read(f'./CORE-COF-2DCOF-1099/{i}.cif')
    gl_chem_feats.append(global_chem_feats(struc))
    #l_chem_feats.append(local_chem_feats(struc))
    nl, cm = compute_ase_neighbour(struc)
    graph = matrix2dict(cm)
    cutoffs = natural_cutoffs(struc, mult=1.0)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(struc)
    G = nx.from_numpy_array(cm)
    #struc analysis
    sp2_o = find_NO(struc)[0]
    sp2_n = find_NO(struc)[1]
    six_rings = find_rings(G, 6)
    five_rings = find_rings(G, 5)
    tot_rings = six_rings + five_rings
    ## mw #
    mw = struc.get_masses().sum()
    mw_ccof.append(mw)
    ###misc feats###
    if len(sp2_o) != 0 or len(sp2_n) !=0:
        nredox = len(sp2_o) + len(sp2_n)
    else:
        nredox = 0.0
    temp = [mw,nredox]
    misc_feats.append(temp)
    ## lattice params#
    a_value = struc.get_cell_lengths_and_angles()[0]
    b_value = struc.get_cell_lengths_and_angles()[1]
    c_value = struc.get_cell_lengths_and_angles()[2]
    lattice_param = [a_value,b_value,c_value]
    lattice_params.append(lattice_param)

    ### special groups
    ##### carboxylate
    n_cooh, c_cooh, o_cooh = identify_carboxylate_groups(struc)
    sp2_o_wo_cooh = [x for x in sp2_o if x not in o_cooh] ## remove carboxyl oxygens
    if n_cooh != 0:
        cofs_cooh.append(i)
    ### sulphonyl
    n_so3h, o_so3h = identify_sulfonic_acid_groups(struc)
    only_sp2_o = [x for x in sp2_o_wo_cooh if x not in o_so3h]
    if n_so3h != 0:
        cofs_so3h.append(i)
    ###bisimcooh
    n_ncoco, motif_ncoco, o_ncoco = identify_bicarboxylic_imides_v2(struc)
    if n_ncoco != 0:
        cofs_ncoco.append(i)

    ### conn
    n_conn, motif_conn, o_conn = identify_conn_motifs(struc)
    if n_conn != 0:
        cofs_conn.append(i)

    ### no2 groups

    n_nitro, motif_nitro, o_nitro = identify_nitro_groups(struc)
    if n_nitro != 0:
        cofs_nitro.append(i)
    ### porphyrin
    n_por, motif_por = identify_porphyrin_units_by_proximity(struc)
    if n_por != 0:
        cofs_por.append(i)
    # PACMOF featurizer
    df = get_features_from_cif_serial(f'./CORE-COF-2DCOF-1099/{i}.cif')
    lcf_o = np.zeros(8)
    lcf_n = np.zeros(8)
    lcf_rings = np.zeros(8)
    w = 0
    x = 0
    if len(only_sp2_o) !=0:
        if n_ncoco == 0 and n_conn == 0:
            mean_feats_o = Agg_mean_feats(only_sp2_o,df)
            lcf_o = lcf_maker(only_sp2_o,mean_feats_o)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,only_sp2_o,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        elif n_ncoco == 0 and n_conn != 0:
            lcf_o = lcf_maker_motif(n_conn,motif_conn,df)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,o_conn,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        elif n_ncoco != 0 and n_conn == 0:
            lcf_o = lcf_maker_motif(n_ncoco,motif_ncoco,df)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,o_ncoco,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        elif n_ncoco != 0 and n_conn != 0: # prioritize NCOCO
            lcf_o = lcf_maker_motif(n_ncoco,motif_ncoco,df)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,o_ncoco,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        else:
            w = w +1
        print('local feats', l_chem_feats)
    elif len(sp2_n)!=0:
        if n_por == 0:
            mean_feats_n = Agg_mean_feats(sp2_n,df)
            lcf_n = lcf_maker(sp2_n,mean_feats_n)
            l_chem_feats.append(lcf_n)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,sp2_n,2.25,10)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_n = count_n + 1
        elif n_por != 0:
            lcf_n = lcf_maker_motif(2*n_por,motif_por,df)
            l_chem_feats.append(lcf_n)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,np.array(motif_por).flatten(),c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_n = count_n + 1
        else:
            x = x + 1
    else:
        lcf_rings[0] = len(tot_rings)
        lcf_rings[1] = 2.275 ## avg of electronegetivity of C and H
        lcf_rings[2] = 12.429 ## avg of 1st IP of C and H
        lcf_rings[3] = 6 ## numer of atoms in 1st coord shell
        lcf_rings[4] = 2.55 ## numer of atoms in 1st coord shell
        lcf_rings[5] = 2.0 ## avg distance from atoms in 1st coord shell
        lcf_rings[6] = 11.260 ## avg 1st IP in 1st coord shell
        lcf_rings[7] = 2.20 ## avg EN  in 2nd coord shell
        #lcf = np.concatenate((lcf_o,lcf_n,lcf_rings), axis=None)
        l_chem_feats.append(lcf_rings)
        #l_chem_feats.append(lcf)
        l_fp_feats.append([0.0, 0.0, 0.0, 10.0, 10.0])
        l_rdf_feats.append(np.zeros([10]))
        count_r = count_r + 1
    print("finished computing features")

/home/sdas/miniconda3/envs/oms/lib/python3.12/site-packages/ase/io/cif.py:401: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!
  warnings.warn(
/home/sdas/miniconda3/envs/oms/lib/python3.12/site-packages/ase/utils/__init__.py:62: FutureWarning: Please use atoms.cell.cellpar() instead
  warnings.warn(warning)


Reading  CIF file ./CORE-COF-2DCOF-1099/885.cif...
Computing features for ./CORE-COF-2DCOF-1099/885.cif...


100%|████████████████████████████████████████████████████████████████████████████████████████████████| 228/228 [00:02<00:00, 81.95it/s]


The neighborlist of 10 is [200  14  48  88 162  78  77  86 200 153 226 154 167 155 162   0   1   2
   3   8]
proximal atom found between 10 and 200
not proximal
proximal atom found between 10 and 48
not proximal
proximal atom found between 10 and 162
not proximal
not proximal
proximal atom found between 10 and 86
proximal atom found between 10 and 200
not proximal
not proximal
not proximal
not proximal
proximal atom found between 10 and 155
proximal atom found between 10 and 162
not proximal
not proximal
not proximal
proximal atom found between 10 and 3
not proximal
[[8, 3.34, 3.44, 11.485278391293914, 2.8736758542079337]]
The neighborlist of 29 is [107  33  96  67  97 105  19  20  21  22  27]
not proximal
not proximal
not proximal
proximal atom found between 29 and 67
not proximal
proximal atom found between 29 and 105
not proximal
not proximal
not proximal
proximal atom found between 29 and 22
not proximal
[[8, 3.34, 3.44, 11.485278391293914, 2.8736758542079337], [3, 3.30666666666666

Reading  CIF file ./CORE-COF-2DCOF-1099/886.cif...
Computing features for ./CORE-COF-2DCOF-1099/886.cif...


...

The neighborlist of 26 is [28 27 29 73 26 20 22 25 26]
proximal atom found between 26 and 28
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
[[1, 3.04, 3.04, 2.6356245513912797, 2.6356245513912797]]
The neighborlist of 57 is [85 60 59 58 57 51 53 56 57]
not proximal
not proximal
proximal atom found between 57 and 59
not proximal
not proximal
not proximal
not proximal
[[1, 3.04, 3.04, 2.6356245513912797, 2.6356245513912797], [1, 3.04, 3.04, 2.6356245513912824, 2.6356245513912824]]
[4.74e-06, 44.87536134, 21.34737113, 3.9e-07, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
local feats [array([12.        ,  3.44      , 13.618     ,  1.        ,  1.21348365,
        2.55      , 11.26      ,  2.55      ]), array([ 6.        ,  3.44      , 13.618     ,  1.        ,  1.19964229,
        2.55      , 11.26      ,  2.55      ]), array([12.        ,  3.44      , 13.618     ,  1.        ,  1.19964229,
        2.55      , 11.26      ,  2.55      ]), array([ 3.        ,  3.04      , 14.

Reading  CIF file ./CORE-COF-2DCOF-1099/899.cif...
Computing features for ./CORE-COF-2DCOF-1099/899.cif...


100%|████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [00:00<00:00, 1374.19it/s]

The neighborlist of 54 is [56 25  3 48 54 19 25  3 48  0 18 18 19 23 24 54]
proximal atom found between 54 and 56
not proximal
not proximal
proximal atom found between 54 and 48
not proximal
not proximal
not proximal
proximal atom found between 54 and 48
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
[[3, 3.1733333333333333, 3.44, 7.677686760021429, 3.236751901763232]]
The neighborlist of 55 is [55  2 17 17 21 22 23 24 26 53 55]
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
proximal atom found between 55 and 53
[[3, 3.1733333333333333, 3.44, 7.677686760021429, 3.236751901763232], [1, 3.04, 3.04, 17.65315970313245, 17.65315970313245]]
The neighborlist of 56 is [25  1 56 20  4  4 19 20 21 26 49 54 56]
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
proximal atom found between 56 and 49
proximal atom found between 56 and 54
[[3, 3.17

KeyboardInterrupt: 

In [ ]:
## assembing features for OCV prediction
data1 = csvDf(gl_chem_feats, ['C','H', 'N', 'O', 'NO', 'halo', 'tdu', 'te'])
data2 = csvDf(l_chem_feats, ['a','b','c','d','e','f','g','h'])
data3 = csvDf(l_fp_feats, ['mean_num','mean_en', 'max_en', 'mean_d', 'max_d'])
data4 = csvDf(l_rdf_feats, ['aa','bb','cc','dd','ee','ff','gg','hh','ii', 'jj'])
data5 = csvDf(lattice_params,['a_value','b_value','c_value'])
data_name = pd.DataFrame(ccofs, columns = ['CCOF_id'])
data = pd.concat([data_name,data1,
                  data2,
                  data3,
                 data4,
                 ], 
                 axis=1)
np.save("core_cof_features-2.npy",data.to_numpy())

In [ ]:
# assembling features for capacity prediction
data1 = csvDf(gl_chem_feats, ['C','H', 'N', 'O', 'NO', 'halo', 'tdu', 'te'])
data3 = csvDf(misc_feats, ['MW','nredox'])
data4 = csvDf(lattice_params,['a_value','b_value','c_value'])
data5 = csvDf(l_rdf_feats, ['aa','bb','cc','dd','ee','ff','gg','hh','ii', 'jj'])
data6 = csvDf(l_chem_feats, ['c1','c2','c3','c4','c5','c6','c7','c8'])
data7 = csvDf(l_fp_feats, ['mean_num','mean_en', 'max_en', 'mean_d', 'max_d'])
data_name = pd.DataFrame(ccofs, columns = ['CCOF_id'])
data = pd.concat([data_name,data1,
                  data3,
                  data4,
                  data5,
                  data6,
                  data7,
                 ], 
                 axis=1)
np.save("core_cof_features_4cap.npy",data.to_numpy())

## Featurizing DICE-C

In [238]:
## dice-c features
gl_chem_feats = []
l_chem_feats = []
l_fp_feats = []
l_rdf_feats = []
count_o = 0
count_n = 0
count_r = 0
cofs_ncoco = []
cofs_conn = []
cofs_cooh = []
cofs_so3h = []
cofs_por=[]
cofs_nitro=[]
c_r = 2.0
B = 100
for i in cofid_lp:
    struc = read(f'./c-predict-3p/cifs/{i}.cif')
    gl_chem_feats.append(global_chem_feats(struc))
    #l_chem_feats.append(local_chem_feats(struc))
    nl, cm = compute_ase_neighbour(struc)
    graph = matrix2dict(cm)
    cutoffs = natural_cutoffs(struc, mult=1.0)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(struc)
    G = nx.from_numpy_array(cm)
    #struc analysis
    sp2_o = find_NO(struc)[0]
    sp2_n = find_NO(struc)[1]
    six_rings = find_rings(G, 6)
    five_rings = find_rings(G, 5)
    tot_rings = six_rings + five_rings
    ### special groups
    ##### carboxylate
    n_cooh, c_cooh, o_cooh = identify_carboxylate_groups(struc)
    sp2_o_wo_cooh = [x for x in sp2_o if x not in o_cooh] ## remove carboxyl oxygens
    if n_cooh != 0:
        cofs_cooh.append(i)
    ### sulphonyl
    n_so3h, o_so3h = identify_sulfonic_acid_groups(struc)
    only_sp2_o = [x for x in sp2_o_wo_cooh if x not in o_so3h]
    if n_so3h != 0:
        cofs_so3h.append(i)
    ###bisimcooh
    n_ncoco, motif_ncoco, o_ncoco = identify_bicarboxylic_imides_v2(struc)
    if n_ncoco != 0:
        cofs_ncoco.append(i)

    ### conn
    n_conn, motif_conn, o_conn = identify_conn_motifs(struc)
    if n_conn != 0:
        cofs_conn.append(i)

    ### no2 groups

    n_nitro, motif_nitro, o_nitro = identify_nitro_groups(struc)
    if n_nitro != 0:
        cofs_nitro.append(i)
    ### porphyrin
    n_por, motif_por = identify_porphyrin_units_by_proximity(struc)
    if n_por != 0:
        cofs_por.append(i)
    # PACMOF featurizer
    df = get_features_from_cif_serial(f'./c-predict-3p/cifs/{i}.cif')
    lcf_o = np.zeros(8)
    lcf_n = np.zeros(8)
    lcf_rings = np.zeros(8)
    w = 0
    x = 0
    #redox_count = 
    if len(only_sp2_o) !=0:
        if n_ncoco == 0 and n_conn == 0:
            mean_feats_o = Agg_mean_feats(only_sp2_o,df)
            lcf_o = lcf_maker(only_sp2_o,mean_feats_o)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,only_sp2_o,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        elif n_ncoco == 0 and n_conn != 0:
            lcf_o = lcf_maker_motif(n_conn,motif_conn,df)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,o_conn,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        elif n_ncoco != 0 and n_conn == 0:
            lcf_o = lcf_maker_motif(n_ncoco,motif_ncoco,df)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,o_ncoco,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        elif n_ncoco != 0 and n_conn != 0: # prioritize NCOCO
            lcf_o = lcf_maker_motif(n_ncoco,motif_ncoco,df)
            l_chem_feats.append(lcf_o)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,o_ncoco,c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_o = count_o + 1
        else:
            w = w +1
        print('local feats', l_chem_feats)
    elif len(sp2_n)!=0:
        if n_por == 0:
            mean_feats_n = Agg_mean_feats(sp2_n,df)
            lcf_n = lcf_maker(sp2_n,mean_feats_n)
            l_chem_feats.append(lcf_n)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,sp2_n,2.25,10)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_n = count_n + 1
        elif n_por != 0:
            lcf_n = lcf_maker_motif(2*n_por,motif_por,df)
            l_chem_feats.append(lcf_n)
            fp_feats, rdf = determin_prox_sites_aprdf(struc,G,np.array(motif_por).flatten(),c_r,B)
            l_fp_feats.append(fp_feats)
            l_rdf_feats.append(rdf)
            count_n = count_n + 1
        else:
            x = x + 1
    else:
        lcf_rings[0] = len(tot_rings)
        lcf_rings[1] = 2.275 ## avg of electronegetivity of C and H
        lcf_rings[2] = 12.429 ## avg of 1st IP of C and H
        lcf_rings[3] = 6 ## numer of atoms in 1st coord shell
        lcf_rings[4] = 2.55 ## numer of atoms in 1st coord shell
        lcf_rings[5] = 2.0 ## avg distance from atoms in 1st coord shell
        lcf_rings[6] = 11.260 ## avg 1st IP in 1st coord shell
        lcf_rings[7] = 2.20 ## avg EN  in 2nd coord shell
        #lcf = np.concatenate((lcf_o,lcf_n,lcf_rings), axis=None)
        l_chem_feats.append(lcf_rings)
        #l_chem_feats.append(lcf)
        l_fp_feats.append([0.0, 0.0, 0.0, 10.0, 10.0])
        l_rdf_feats.append(np.zeros([10]))
        count_r = count_r + 1
    print("finished computing features")

Reading  CIF file ./c-predict-3p/cifs/1049.cif...
Computing features for ./c-predict-3p/cifs/1049.cif...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 120/120 [00:00<00:00, 560.65it/s]


The neighborlist of 4 is [ 34  40 104  16  17   4   1   2   3   4]
not proximal
proximal atom found between 4 and 40
not proximal
not proximal
not proximal
not proximal
not proximal
proximal atom found between 4 and 3
[[2, 3.24, 3.44, 2.3165661206533255, 2.316566120653325]]
The neighborlist of 20 is [33 96 32 20  2  8 17 18 19 20]
not proximal
not proximal
not proximal
not proximal
proximal atom found between 20 and 8
not proximal
not proximal
proximal atom found between 20 and 19
[[2, 3.24, 3.44, 2.3165661206533255, 2.316566120653325], [2, 3.24, 3.44, 2.3165661206533255, 2.3165661206533197]]
The neighborlist of 36 is [100  36   0   1  18  24  33  34  35  36]
not proximal
not proximal
not proximal
not proximal
proximal atom found between 36 and 24
not proximal
not proximal
proximal atom found between 36 and 35
[[2, 3.24, 3.44, 2.3165661206533255, 2.316566120653325], [2, 3.24, 3.44, 2.3165661206533255, 2.3165661206533197], [2, 3.24, 3.44, 2.3165661206533272, 2.3165661206533255]]
The nei

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 108/108 [00:00<00:00, 297.36it/s]


The neighborlist of 26 is [27 31 29 28 30 26 18 21 22 24 26]
not proximal
proximal atom found between 26 and 29
proximal atom found between 26 and 28
not proximal
not proximal
not proximal
not proximal
not proximal
[[2, 3.24, 3.44, 3.677507461714696, 2.840070235290142]]
The neighborlist of 28 is [38 32 29 28 20 23 25 26 27 28]
not proximal
not proximal
proximal atom found between 28 and 29
not proximal
not proximal
not proximal
proximal atom found between 28 and 26
[[2, 3.24, 3.44, 3.677507461714696, 2.840070235290142], [2, 3.24, 3.44, 3.5602285633868043, 2.605512438634358]]
The neighborlist of 62 is [64 63 65 66 67 62 54 57 58 60 62]
proximal atom found between 62 and 64
proximal atom found between 62 and 65
not proximal
not proximal
not proximal
not proximal
not proximal
not proximal
[[2, 3.24, 3.44, 3.677507461714696, 2.840070235290142], [2, 3.24, 3.44, 3.5602285633868043, 2.605512438634358], [2, 3.24, 3.44, 3.6775074617146997, 2.8400702352901477]]
The neighborlist of 64 is [68 65 7

...

finished computing features
Reading  CIF file ./c-predict-3p/cifs/463.cif...
Computing features for ./c-predict-3p/cifs/463.cif...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 78/78 [00:00<00:00, 1195.15it/s]


finished computing features
Reading  CIF file ./c-predict-3p/cifs/466.cif...
Computing features for ./c-predict-3p/cifs/466.cif...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [00:00<00:00, 757.58it/s]

finished computing features


In [239]:
## additional features
gl_chem_feats = []
misc_feats = []
mw_lp = np.array(mw_lp)
for i in range(len(cofid_lp)):
    struc = read(f'./c-predict-3p/cifs/{cofid_lp[i]}.cif')
    gl_chem_feats.append(global_chem_feats(struc))
    sp2_o = find_NO(struc)[0]
    sp2_n = find_NO(struc)[1]
    nl, cm = compute_ase_neighbour(struc)
    graph = matrix2dict(cm)
    cutoffs = natural_cutoffs(struc, mult=1.0)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True)
    nl.update(struc)
    G = nx.from_numpy_array(cm)
    six_rings = find_rings(G, 6)
    five_rings = find_rings(G, 5)
    tot_rings = six_rings + five_rings
    if len(sp2_o) != 0 or len(sp2_n) !=0:
        nredox = len(sp2_o) + len(sp2_n)
    else:
        nredox = 0.0
    temp = [mw_lp[i],nredox]
    misc_feats.append(temp)

In [ ]:
data1 = csvDf(gl_chem_feats, ['C','H', 'N', 'O', 'NO', 'halo', 'tdu', 'te'])
data2 = csvDf(sfeats_lp, ['a','b','c','d','e','f','g'])
data3 = csvDf(misc_feats, ['MW','nredox'])
data4 = csvDf(lattice_params,['a_value','b_value','c_value'])
data5 = csvDf(l_rdf_feats, ['aa','bb','cc','dd','ee','ff','gg','hh','ii', 'jj'])
data6 = csvDf(l_chem_feats, ['c1','c2','c3','c4','c5','c6','c7','c8'])
data7 = csvDf(l_fp_feats, ['mean_num','mean_en', 'max_en', 'mean_d', 'max_d'])
labels = []
for i in cap_lp:
    if  i >= 200:
        labels.append(1)
    else:
        labels.append(0)
data_t = pd.DataFrame(labels, columns = ['class'])
data = pd.concat([data1,
                  #data2,
                  data3,
                  data4,
                  data5,
                  data6,
                  data7,
                  data_t,
                 ], 
                 axis=1)
np.save("cap_data_cls.npy",data.to_numpy())